# MR3b — Hourly Final Join (Reduce-Side)
**Inputs**: `mr3/hourly_crash_mr.csv` + `mr3/address_store_mr.csv` + `mr3/type_count_store_mr.csv`
**Output**: `mr3/mr_hourly_final.csv`

Join type: **inner** on `zone_id` (reduce-side join)
Mapper tags each record by source. Shuffle groups by zone_id. Reducer merges crash + store records.
Crash-only zones (no matching store) are dropped.

In [1]:
import json
import pandas as pd
from collections import defaultdict

CRASH_INPUT  = 'hourly_crash_mr.csv'
ADDR_INPUT   = 'address_store_mr.csv'
TYPE_INPUT   = 'type_count_store_mr.csv'
OUTPUT       = 'mr_hourly_final.csv'

crashes = pd.read_csv(CRASH_INPUT)
addr    = pd.read_csv(ADDR_INPUT)
types   = pd.read_csv(TYPE_INPUT)

print(f'Hourly crash rows: {len(crashes):,}  |  zones: {crashes["zone_id"].nunique():,}')
print(f'Store zones      : {len(addr):,}')
print(f'Type rows        : {len(types):,}')

Hourly crash rows: 2,708  |  zones: 1,490
Store zones      : 1,823
Type rows        : 6,576


In [2]:
# ── MAP ───────────────────────────────────────────────────────────────────────
# Tag each record with its source, emit (zone_id, tagged_record)

# Collapse type rows into one dict per zone before mapping
type_by_zone = defaultdict(dict)
for _, row in types.iterrows():
    type_by_zone[str(row['zone_id'])][row['method_of_operation']] = int(row['count'])

def mapper(row, source):
    zid = str(row['zone_id'])
    if source == 'crash':
        return zid, {
            'src':        'crash',
            'crash_date': row['crash_date'],  # or 'hour' for notebook 6
            'crashes':    row['crashes'],
            'injured':    row['injured'],
            'killed':     row['killed'],
        }
    else:  # store
        return zid, {
            'src':               'store',
            'store_count':       row['store_count'],
            'active_licenses':   row['active_licenses'],
            'outdated_licenses': row['outdated_licenses'],
            'type_counts_json':  json.dumps(type_by_zone.get(zid, {})),
        }

mapped  = [mapper(row, 'crash') for _, row in crashes.iterrows()]
mapped += [mapper(row, 'store') for _, row in addr.iterrows()]

print(f'Total mapped pairs: {len(mapped):,}')

Total mapped pairs: 4,531


In [3]:
# ── SHUFFLE ───────────────────────────────────────────────────────────────────
# Group all tagged records by zone_id

shuffled = defaultdict(list)
for zone_id, record in mapped:
    shuffled[zone_id].append(record)

print(f'Unique zone_ids after shuffle: {len(shuffled):,}')

Unique zone_ids after shuffle: 2,289


In [4]:
# ── REDUCE ────────────────────────────────────────────────────────────────────
# For each zone_id: find store record, join with all crash records.
# Zones with no matching store are dropped (crash-only zones).

def reducer(zone_id, records):
    store_rec  = next((r for r in records if r['src'] == 'store'), None)
    crash_recs = [r for r in records if r['src'] == 'crash']
    if not store_rec or not crash_recs:
        return []
    return [
        {
            'zone_id':           zone_id,
            'hour':              c['hour'],
            'avg_crashes':       c['avg_crashes'],
            'avg_injured':       c['avg_injured'],
            'avg_killed':        c['avg_killed'],
            'store_count':       store_rec['store_count'],
            'active_licenses':   store_rec['active_licenses'],
            'outdated_licenses': store_rec['outdated_licenses'],
            'type_counts_json':  store_rec['type_counts_json'],
        }
        for c in crash_recs
    ]

rows = []
for zone_id, records in shuffled.items():
    rows.extend(reducer(zone_id, records))

out = pd.DataFrame(rows).sort_values(['zone_id','hour']).reset_index(drop=True)
out.to_csv(OUTPUT, index=False)

print(f'Output rows  : {len(out):,}')
print(f'Unique zones : {out["zone_id"].nunique():,}')
print(f'Hours covered: {sorted(out["hour"].unique())}')
print(f'Dropped crash-only zones: {crashes["zone_id"].nunique() - out["zone_id"].nunique():,}')
print(f'Saved → {OUTPUT}')
out.head()

Output rows  : 1,973
Unique zones : 1,024
Hours covered: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23)]
Dropped crash-only zones: 466
Saved → mr_hourly_final.csv


,zone_id,hour,avg_crashes,avg_injured,avg_killed,store_count,active_licenses,outdated_licenses,type_counts_json
0,8a2a1008804ffff,6,1.0,0.0,0.0,3,0,3,"{""Tavern Serving Beer Cider Wine And Liquor"": ..."
1,8a2a1008804ffff,21,1.0,1.0,0.0,3,0,3,"{""Tavern Serving Beer Cider Wine And Liquor"": ..."
2,8a2a1008814ffff,4,1.0,1.0,0.0,6,1,5,"{""Restaurant Serving Beer, Cider And Wine"": 1,..."
3,8a2a1008814ffff,9,1.0,0.0,0.0,6,1,5,"{""Restaurant Serving Beer, Cider And Wine"": 1,..."
4,8a2a1008814ffff,17,1.0,0.0,0.0,6,1,5,"{""Restaurant Serving Beer, Cider And Wine"": 1,..."
